# Naive Bayes With Natural Language Processing

In this demo, we will use scikit-learn's text processing pipeline to perform Natural Language Processing (NLP) on a data set and generate a Naive Bayes model which performs sentiment analysis.

We will use:
* `MultinomialNB` from scikit-learn for our Naive Bayes classifier
* `TfidfVectorizer` to convert text into numerical features
* `Pipeline` to chain the vectorizer and classifier together
* `ConfusionMatrixDisplay` for visualization

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

Our data set is a sample of 2000 movie reviews. There are two variables in this data set: a **class** feature which is "Pos" (for a positive review) and "Neg" (for a negative review). Then, **text** is the review itself.

In [ ]:
df = pd.read_csv("../data/movie-pang02.csv")
df.head()

In [ ]:
df.info()

Like the first demo, we shuffle the data and split into training and test sets. We use 75% for training and 25% for testing, stratified by class to ensure balanced splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["class"], test_size=0.25, random_state=1, stratify=df["class"]
)
print(f"Training set: {len(X_train)} reviews")
print(f"Test set: {len(X_test)} reviews")

## Approach 1: Bag of Words with BernoulliNB

We use `CountVectorizer` to build a document-term matrix, binarize it (presence/absence of a term rather than count), and feed it to `BernoulliNB`.

`CountVectorizer` handles text preprocessing for us — converting to lowercase, removing punctuation, and tokenizing. We set `min_df=5` to keep only terms appearing in at least 5 documents. Setting `binary=True` converts counts to presence/absence, focusing on word usage rather than frequency.

In [ ]:
bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        stop_words="english",
        min_df=5,
        binary=True
    )),
    ("classifier", BernoulliNB(alpha=1.0))  # alpha=1.0 is Laplace smoothing
])

bow_pipeline.fit(X_train, y_train)
bow_predictions = bow_pipeline.predict(X_test)

In [ ]:
vocab_size = len(bow_pipeline.named_steps["vectorizer"].vocabulary_)
print(f"Vocabulary size (terms appearing in 5+ documents): {vocab_size}")

Now let's look at the confusion matrix and classification report.

**Sensitivity (Recall)** is where we capture when an event is positive, whether our predictor considers it positive.

**Specificity** is where we capture when an event is negative, whether our predictor considers it negative.

**Positive predictive value (Precision)** looks at all cases where the prediction was positive — of those, how many were actually positive?

**Negative predictive value** looks at cases where the prediction was negative — of those, how many were actually negative?

In [ ]:
print(classification_report(y_test, bow_predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, bow_predictions, cmap="Blues")
plt.title("Bag of Words + BernoulliNB")
plt.show()

## Approach 2: TF-IDF with MultinomialNB

Now let's try TF-IDF (Term Frequency-Inverse Document Frequency) weighting with `MultinomialNB`. TF-IDF gives higher weight to terms that are distinctive to a document rather than common across all documents.

scikit-learn's `Pipeline` makes this clean — swap the vectorizer and classifier.

In [ ]:
tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(stop_words="english", min_df=5)),
    ("classifier", MultinomialNB(alpha=1.0))
])

tfidf_pipeline.fit(X_train, y_train)
tfidf_predictions = tfidf_pipeline.predict(X_test)

In [ ]:
print(classification_report(y_test, tfidf_predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, tfidf_predictions, cmap="Blues")
plt.title("TF-IDF + MultinomialNB")
plt.show()

## Comparing the Two Approaches

Let's compare accuracy side by side.

In [ ]:
from sklearn.metrics import accuracy_score

print(f"Bag of Words + BernoulliNB accuracy: {accuracy_score(y_test, bow_predictions):.4f}")
print(f"TF-IDF + MultinomialNB accuracy:     {accuracy_score(y_test, tfidf_predictions):.4f}")